In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install    mne

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 46.9 MB/s eta 0:00:00
  Attempting uninstall: decorator
    Found existing installation: decorator 4.4.2
    Uninstalling decorator-4.4.2:
      Successfully uninstalled decorator-4.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [ ]:
import os, glob, random, gc
import numpy as np
import pandas as pd
import mne
import scipy.signal
import scipy.stats
from tqdm.auto import tqdm
import warnings
from collections import Counter

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')


TARGET_CLASSES = ['fnsz', 'gnsz', 'cpsz']

CHANNELS_ORDER = [
    'FP1', 'FP2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4',
    'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6',
    'A1', 'A2', 'FZ', 'CZ'
]

WINDOW_SIZE_SEC   = 10.0
TARGET_SFREQ      = 128.0
POINTS_PER_WINDOW = int(WINDOW_SIZE_SEC * TARGET_SFREQ)


CLASS_STRIDES_SEC = {
    'cpsz': 1.0,
    'gnsz': 4.0,
    'fnsz': 4.0
}


DATASET_ROOT = '/content/drive/MyDrive/gp_dataset/seizure_v2'


SAVE_DIR = '/content/drive/MyDrive/gp_dataset/classification_v2_split_PSD_features_1'
os.makedirs(SAVE_DIR, exist_ok=True)

def extract_frequency_maps_vectorized(batch_data, sfreq=128.0, max_batch_size=500):
    B_total, C, T_pts = batch_data.shape
    num_bands = 5

    all_spatial_maps = []

    for i in range(0, B_total, max_batch_size):
        mb_data = batch_data[i : i + max_batch_size]
        B = mb_data.shape[0]

        freqs, psd = scipy.signal.welch(mb_data, sfreq, nperseg=256, axis=-1)

        bands = {
            'delta': (0.5, 4),
            'theta': (4, 8),
            'alpha': (8, 13),
            'beta': (13, 30),
            'gamma': (30, 40)
        }

        powers = []
        for low, high in bands.values():
            idx_band = np.logical_and(freqs >= low, freqs <= high)
            powers.append(np.trapz(psd[:, :, idx_band], freqs[idx_band], axis=-1))

        psd_features = np.stack(powers, axis=-1)

        spatial_map = np.zeros((B, 5, 5, num_bands))

        mapping = {
            (0,1): 0,  (0,2): 18, (0,3): 1,
            (1,0): 10, (1,1): 2,  (1,3): 3,  (1,4): 11,
            (2,0): 12, (2,1): 4,  (2,2): 19, (2,3): 5,  (2,4): 13,
            (3,0): 14, (3,1): 6,  (3,3): 7,  (3,4): 15,
            (4,0): 16, (4,1): 8,  (4,3): 9,  (4,4): 17
        }

        for (row, col), ch_idx in mapping.items():
            spatial_map[:, row, col, :] = psd_features[:, ch_idx, :]

        mean_val = np.mean(spatial_map, axis=(1,2,3), keepdims=True)
        std_val = np.std(spatial_map, axis=(1,2,3), keepdims=True)
        spatial_map = (spatial_map - mean_val) / (std_val + 1e-8)

        all_spatial_maps.append(spatial_map)

        del mb_data, psd, powers, psd_features, spatial_map
        gc.collect()

    return np.concatenate(all_spatial_maps, axis=0)

def process_single_csv(csv_path):
    raw_windows_list, labels_list = [], []
    MAX_WINDOWS_PER_FILE = 2000

    try:
        edf_path = csv_path.replace('.csv', '.edf').replace('.csv.tse', '.edf')
        if not os.path.exists(edf_path): return [], [], []

        df_ann = pd.read_csv(csv_path, skiprows=6, header=None)
        seizure_events = [(float(r[1]), float(r[2]), str(r[3]).lower().strip())
                          for _, r in df_ann.iterrows() if str(r[3]).lower().strip() in TARGET_CLASSES]

        if not seizure_events: return [], [], []

        raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
        actual_picks = [raw.ch_names.index(c) for target in CHANNELS_ORDER for c in raw.ch_names if f"EEG {target}-" in c]
        if len(actual_picks) != len(CHANNELS_ORDER): return [], [], []

        original_sfreq = raw.info['sfreq']

        for (s_start, s_end, s_label) in seizure_events:
            if len(labels_list) >= MAX_WINDOWS_PER_FILE:
                break

            start_idx = int(s_start * original_sfreq)
            end_idx   = int(s_end   * original_sfreq)

            if (end_idx - start_idx) < (WINDOW_SIZE_SEC * original_sfreq):
                continue

            data, _ = raw[actual_picks, start_idx:end_idx]
            if data.shape[1] == 0: continue

            if original_sfreq > TARGET_SFREQ:
                data = mne.filter.resample(data, up=TARGET_SFREQ, down=original_sfreq)
                sfreq = TARGET_SFREQ
            else:
                sfreq = original_sfreq

            data = mne.filter.notch_filter(data, sfreq, freqs=60.0, verbose=False)
            data = mne.filter.filter_data(data, sfreq, l_freq=0.5, h_freq=40.0, verbose=False)

            median_val = np.median(data, axis=1, keepdims=True)
            q75, q25 = np.percentile(data, [75 ,25], axis=1, keepdims=True)
            iqr_val = q75 - q25
            data = (data - median_val) / (iqr_val + 1e-8)
            data = np.clip(data, -5.0, 5.0)

            stride_sec = CLASS_STRIDES_SEC.get(s_label, 2.0)
            stride_points = int(stride_sec * TARGET_SFREQ)

            for win_start in range(0, data.shape[1] - POINTS_PER_WINDOW + 1, stride_points):
                if len(labels_list) >= MAX_WINDOWS_PER_FILE:
                    break

                window_data = data[:, win_start:win_start + POINTS_PER_WINDOW]
                raw_windows_list.append(window_data)
                labels_list.append(s_label)

            del data
            gc.collect()

        del raw
        gc.collect()

    except Exception as e:
        pass

    if not raw_windows_list:
        return [], [], []

    raw_windows_arr = np.stack(raw_windows_list)
    del raw_windows_list
    gc.collect()

    X_eng_local = extract_frequency_maps_vectorized(raw_windows_arr, sfreq=TARGET_SFREQ)
    X_raw_local = np.swapaxes(raw_windows_arr, 1, 2)

    return X_raw_local, X_eng_local, labels_list


def extract_for_direct_folder(split_name, chunk_size=3):
    print(f"\nProcessing split: {split_name.upper()}...")


    target_folder = os.path.join(DATASET_ROOT, split_name.lower())
    split_csvs = glob.glob(os.path.join(target_folder, '*.csv'))

    total_chunks = (len(split_csvs) // chunk_size) + 1

    for chunk_idx, i in enumerate(range(0, len(split_csvs), chunk_size)):

        if os.path.exists(f'{SAVE_DIR}/y_{split_name.lower()}_part{chunk_idx}.npy'):
            print(f"    Skipping chunk {chunk_idx + 1}/{total_chunks} (Already exists)...")
            continue

        chunk_csvs = split_csvs[i:i+chunk_size]
        print(f"\n    Processing chunk {chunk_idx + 1}/{total_chunks} ({len(chunk_csvs)} files)...")

        X_raw_list, X_eng_list, y_list = [], [], []

        for csv in tqdm(chunk_csvs, desc=f"Part {chunk_idx+1}", leave=False):
            try:
                raw_res, eng_res, labels_res = process_single_csv(csv)
                if len(labels_res) > 0:
                    X_raw_list.append(raw_res)
                    X_eng_list.append(eng_res)
                    y_list.extend(labels_res)
            except Exception:
                continue
            gc.collect()

        if len(X_raw_list) > 0:
            X_raw_chunk = np.concatenate(X_raw_list, axis=0).astype(np.float32)
            X_eng_chunk = np.concatenate(X_eng_list, axis=0).astype(np.float32)
            y_chunk = np.array(y_list)

            np.save(f'{SAVE_DIR}/X_raw_{split_name.lower()}_part{chunk_idx}.npy', X_raw_chunk)
            np.save(f'{SAVE_DIR}/X_eng_{split_name.lower()}_part{chunk_idx}.npy', X_eng_chunk)
            np.save(f'{SAVE_DIR}/y_{split_name.lower()}_part{chunk_idx}.npy', y_chunk)

            print(f"    Saved Part {chunk_idx} -> Raw: {X_raw_chunk.shape} | Eng: {X_eng_chunk.shape}")

            del X_raw_chunk, X_eng_chunk, y_chunk, X_raw_list, X_eng_list, y_list
            gc.collect()


extract_for_direct_folder("TRAIN", chunk_size=3)
extract_for_direct_folder("VAL", chunk_size=3)
extract_for_direct_folder("TEST", chunk_size=3)

print(f"\nDone! All features extracted and saved into:\n{SAVE_DIR}")


Processing split: TRAIN...

    Processing chunk 1/1322 (3 files)...


Part 1:   0%|          | 0/3 [00:00<?, ?it/s]


    Processing chunk 2/1322 (3 files)...


Part 2:   0%|          | 0/3 [00:00<?, ?it/s]

    Saved Part 1 -> Raw: (8, 1280, 20) | Eng: (8, 5, 5, 5)

    Processing chunk 3/1322 (3 files)...


...


    Processing chunk 217/218 (3 files)...


Part 217:   0%|          | 0/3 [00:00<?, ?it/s]


    Processing chunk 218/218 (2 files)...


Part 218:   0%|          | 0/2 [00:00<?, ?it/s]


Done! All features extracted and saved into:
/content/drive/MyDrive/gp_dataset/classification_v2_split_PSD_features_1
